[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/kestra-certified/notebooks/day-07-subflows-namespaces.ipynb#scrollTo=a1b2c3d4)

---
# Day 7 · Subflows and Namespaces — Modular Workflow Design
**certified-journeys / kestra-certified** · Week 2 · Modular Orchestration

> **Goal for today:** Design reusable Kestra subflows with typed inputs, organize flows into a namespace hierarchy, and reference namespace files from Python tasks using Pebble templates.


In [ ]:
%pip install -q pyyaml


## Step 1 · What is a Subflow?

A **subflow** in Kestra is a flow called from another flow — just like calling a function from code. The parent flow passes **inputs**; the child flow returns **outputs**. The child runs as an independent execution with its own logs and state.

Key characteristics:
- Called with `io.kestra.plugin.core.flow.Subflow`
- The `flowId` and `namespace` of the child are required
- `inputs` map to the child flow's declared `inputs` block
- `outputs` from the child are accessible via `{{ outputs.myTask.childTaskOutput }}`

**When to extract a subflow:**
- A task sequence is reused across 2+ parent flows
- A logical unit has its own retry / error handling needs
- You want independent scheduling of a sub-process

| Concept | Analogy |
|---|---|
| Subflow | Function call |
| Subflow inputs | Function parameters |
| Subflow outputs | Return values |
| Namespace | Python module / package path |


In [ ]:
import yaml

# ── Child flow: a reusable data-quality check ─────────────────────────────────
child_flow = {
    "id": "check-row-count",
    "namespace": "data.validation",
    "description": "Validates that a dataset has at least min_rows rows.",
    "inputs": [
        {"id": "dataset_name", "type": "STRING"},
        {"id": "row_count",    "type": "INT"},
        {"id": "min_rows",     "type": "INT", "defaults": 1},
    ],
    "tasks": [
        {
            "id": "validate",
            "type": "io.kestra.plugin.core.log.Log",
            "message": (
                "Checking {{ inputs.dataset_name }}: "
                "{{ inputs.row_count }} rows (min={{ inputs.min_rows }})"
            ),
        },
        {
            "id": "fail-if-empty",
            "type": "io.kestra.plugin.core.flow.If",
            "condition": "{{ inputs.row_count < inputs.min_rows }}",
            "then": [
                {
                    "id": "raise-error",
                    "type": "io.kestra.plugin.core.execution.Fail",
                    "errorMessage": "Dataset {{ inputs.dataset_name }} has too few rows!",
                }
            ],
        },
    ],
    "outputs": [
        {
            "id": "validation_passed",
            "type": "BOOLEAN",
            "value": "{{ inputs.row_count >= inputs.min_rows }}",
        }
    ],
}

print("=== Child Flow YAML ===")
print(yaml.dump(child_flow, default_flow_style=False, sort_keys=False))


**What just happened?**
- We modeled a **child flow** as a plain Python dict and rendered it to YAML — exactly what you'd paste into the Kestra UI.
- **`inputs` block** declares typed parameters: `STRING`, `INT`, with optional `defaults`.
- **`outputs` block** exposes a computed value that the parent can read back.
- The `io.kestra.plugin.core.flow.If` task branches on a Pebble condition — no scripting needed.


## Step 2 · Calling a Subflow from a Parent Flow

The `io.kestra.plugin.core.flow.Subflow` task is the bridge between parent and child.

Required fields:
```
type: io.kestra.plugin.core.flow.Subflow
namespace: <child's namespace>
flowId: <child's id>
inputs:   # map matching the child's declared inputs
  key: value
wait: true   # block until the child finishes (default: false)
transmitFailed: true  # propagate child failure to parent
```

After the subflow task, access child outputs with:
```
{{ outputs.mySubflowTask.outputs.validation_passed }}
```


In [ ]:
# ── Parent flow calling the child ─────────────────────────────────────────────
parent_flow = {
    "id": "daily-ingest-pipeline",
    "namespace": "data.ingestion",
    "description": "Downloads a dataset then validates it via a subflow.",
    "tasks": [
        {
            "id": "download",
            "type": "io.kestra.plugin.core.http.Download",
            "uri": "https://people.sc.fsu.edu/~jburkardt/data/csv/addresses.csv",
        },
        {
            "id": "count-rows",
            "type": "io.kestra.plugin.core.log.Log",
            # In a real flow you'd use a Python script to count rows;
            # here we simulate a known count for demonstration.
            "message": "Row count computed: 6",
        },
        {
            "id": "validate-dataset",
            "type": "io.kestra.plugin.core.flow.Subflow",
            "namespace": "data.validation",
            "flowId": "check-row-count",
            "inputs": {
                "dataset_name": "addresses.csv",
                "row_count": 6,
                "min_rows": 1,
            },
            "wait": True,           # block parent until child completes
            "transmitFailed": True, # fail parent if child fails
        },
        {
            "id": "log-result",
            "type": "io.kestra.plugin.core.log.Log",
            "message": (
                "Validation result: "
                "{{ outputs.validate-dataset.outputs.validation_passed }}"
            ),
        },
    ],
}

print("=== Parent Flow YAML ===")
print(yaml.dump(parent_flow, default_flow_style=False, sort_keys=False))


**What just happened?**
- **`wait: true`** makes the parent block — without it, the parent fires the child and moves on immediately.
- **`transmitFailed: true`** ensures a child failure surfaces in the parent's execution log.
- The `outputs` reference uses the **task id** (`validate-dataset`), then `.outputs.<output_id>`.
- The parent flow lives in `data.ingestion`; the child lives in `data.validation` — namespaces keep them cleanly separated.


## Step 3 · Namespace Hierarchy

Namespaces in Kestra are **dot-separated paths** — conceptually identical to Python package paths. They control:
- **Flow organization** in the Kestra UI (tree view)
- **Namespace-level variables** (inherited down the tree)
- **Namespace files** (shared scripts / configs stored in Kestra's file store)
- **Access control** — you can restrict which users or service accounts can trigger flows in a namespace

**Recommended hierarchy for a data engineering project:**

```
data
├── data.ingestion         ← HTTP downloads, API pulls, DB extracts
├── data.transformation    ← pandas, dbt, Spark transforms
├── data.validation        ← schema checks, row counts, null checks
└── data.reporting         ← exports, dashboards, alert notifications
```

A flow in `data.transformation` can call a subflow in `data.validation` — namespaces are not silos, just labels.


In [ ]:
# ── Simulate a namespace registry for a data project ─────────────────────────
namespace_registry = {
    "data.ingestion": [
        "daily-ingest-pipeline",
        "ingest-api-events",
        "backfill-historical",
    ],
    "data.transformation": [
        "transform-addresses",
        "aggregate-daily-sales",
    ],
    "data.validation": [
        "check-row-count",
        "check-schema",
        "check-null-pct",
    ],
    "data.reporting": [
        "export-summary-csv",
        "log-daily-report",
    ],
}

print("Namespace tree:")
for ns, flows in namespace_registry.items():
    print(f"  {ns}/")
    for f in flows:
        print(f"    └─ {f}")

# Show how a subflow call looks for each reusable validation flow
print("\nSubflow call example for 'check-schema':")
subflow_call = {
    "id": "validate-schema",
    "type": "io.kestra.plugin.core.flow.Subflow",
    "namespace": "data.validation",
    "flowId": "check-schema",
    "inputs": {"expected_columns": "id,name,city", "dataset_uri": "{{ outputs.download.uri }}"},
    "wait": True,
    "transmitFailed": True,
}
print(yaml.dump(subflow_call, default_flow_style=False, sort_keys=False))


**What just happened?**
- We printed the full namespace tree showing how flows are grouped by responsibility.
- **`data.validation` flows are shared utilities** — any ingestion or transformation flow can call them as subflows without duplicating logic.
- The `{{ outputs.download.uri }}` Pebble expression passes the downloaded file URI from a prior task into the subflow input.


## Step 4 · Namespace Files and `{{ flow.namespace }}`

**Namespace files** are files stored in Kestra's internal file store, scoped to a namespace. They are typically:
- Python helper scripts (`utils.py`, `schema.json`)
- SQL templates
- Configuration files

You upload them via the Kestra UI or API, and reference them in script tasks using:
```yaml
namespaceFiles:
  enabled: true
```
Inside the script, the files are available at their original paths relative to the working directory.

The **`{{ flow.namespace }}`** Pebble variable exposes the current flow's namespace at runtime — useful for dynamic logging or when a script needs to know which environment it's running in.

| Pebble variable | Value at runtime |
|---|---|
| `{{ flow.namespace }}` | `data.ingestion` |
| `{{ flow.id }}` | `daily-ingest-pipeline` |
| `{{ execution.id }}` | unique execution UUID |
| `{{ trigger.date }}` | schedule date (if triggered by Schedule) |


In [ ]:
# ── Flow using a namespace file in a Python script task ──────────────────────
flow_with_namespace_file = {
    "id": "transform-with-utils",
    "namespace": "data.transformation",
    "tasks": [
        {
            "id": "run-transform",
            "type": "io.kestra.plugin.scripts.python.Script",
            # namespaceFiles: enabled=true mounts all files stored under
            # the 'data.transformation' namespace into the working dir.
            "namespaceFiles": {"enabled": True},
            "script": (
                "import utils  # utils.py uploaded to the namespace file store\n"
                "import os\n\n"
                "# flow.namespace is rendered by Kestra before the script runs\n"
                "namespace = '{{ flow.namespace }}'\n"
                "flow_id   = '{{ flow.id }}'\n"
                "exec_id   = '{{ execution.id }}'\n\n"
                "print(f'Running {flow_id} in namespace {namespace}')\n"
                "print(f'Execution: {exec_id}')\n\n"
                "result = utils.transform_data()\n"
                "print(f'Rows transformed: {result[\"rows\"]}')\n"
            ),
        }
    ],
}

print("=== Flow with Namespace File ===")
print(yaml.dump(flow_with_namespace_file, default_flow_style=False, sort_keys=False))

# ── Simulate Pebble rendering locally ────────────────────────────────────────
print("\n=== Simulated Pebble rendering ===")
pebble_vars = {
    "flow.namespace": "data.transformation",
    "flow.id": "transform-with-utils",
    "execution.id": "abcd1234-exec",
}

raw_script = (
    "namespace = '{{ flow.namespace }}'\n"
    "flow_id   = '{{ flow.id }}'\n"
    "exec_id   = '{{ execution.id }}'"
)

rendered = raw_script
for key, val in pebble_vars.items():
    rendered = rendered.replace("{{ " + key + " }}", val)

print("Before rendering:")
print(raw_script)
print("\nAfter Pebble rendering:")
print(rendered)


**What just happened?**
- **`namespaceFiles: enabled: true`** tells Kestra to mount all files stored under the flow's namespace into the script's working directory — so `import utils` just works.
- We simulated **Pebble template rendering** in Python to show how `{{ flow.namespace }}` is substituted before the script executes.
- **`{{ execution.id }}`** is useful for writing output files with unique names (e.g., `results_{execution.id}.csv`).


## Step 5 · Putting It Together — A Three-Namespace Pipeline

Here we compose a complete pipeline that spans three namespaces:
1. `data.ingestion` — downloads a CSV
2. `data.validation` — validates row count (called as subflow)
3. `data.reporting` — logs the summary

This mirrors real-world modular Kestra project structure.


In [ ]:
# ── Compose the three-namespace pipeline ─────────────────────────────────────
full_pipeline = {
    "id": "end-to-end-pipeline",
    "namespace": "data.ingestion",
    "description": "Ingest → Validate → Report across three namespaces.",
    "inputs": [
        {"id": "source_url", "type": "STRING",
         "defaults": "https://people.sc.fsu.edu/~jburkardt/data/csv/addresses.csv"},
        {"id": "min_rows",   "type": "INT",    "defaults": 1},
    ],
    "tasks": [
        # ── Step 1: Ingest ──────────────────────────────────────────────────
        {
            "id": "download",
            "type": "io.kestra.plugin.core.http.Download",
            "uri": "{{ inputs.source_url }}",
        },
        # ── Step 2: Validate (subflow in data.validation) ───────────────────
        {
            "id": "validate",
            "type": "io.kestra.plugin.core.flow.Subflow",
            "namespace": "data.validation",
            "flowId": "check-row-count",
            "inputs": {
                "dataset_name": "{{ inputs.source_url | split('/') | last }}",
                "row_count": 6,
                "min_rows": "{{ inputs.min_rows }}",
            },
            "wait": True,
            "transmitFailed": True,
        },
        # ── Step 3: Report (subflow in data.reporting) ──────────────────────
        {
            "id": "report",
            "type": "io.kestra.plugin.core.flow.Subflow",
            "namespace": "data.reporting",
            "flowId": "log-daily-report",
            "inputs": {
                "message": (
                    "Pipeline complete. Validation passed: "
                    "{{ outputs.validate.outputs.validation_passed }}"
                )
            },
            "wait": True,
            "transmitFailed": False,  # don't fail if reporting fails
        },
    ],
}

print("=== Full Multi-Namespace Pipeline YAML ===")
print(yaml.dump(full_pipeline, default_flow_style=False, sort_keys=False))


**What just happened?**
- **Three namespaces, one parent flow** — the parent only knows the interface (inputs/outputs) of each subflow, not their internals.
- `| split('/') | last` is a Pebble filter chain — it extracts the filename from a URL (`addresses.csv`).
- **`transmitFailed: false`** on the reporting step means a report failure doesn't mark the whole pipeline as failed — a pragmatic production choice.


In [ ]:
# Challenge: Build a subflow for schema validation
#
# Create a child flow YAML for 'check-schema' in namespace 'data.validation'.
# It should accept:
#   - dataset_name (STRING)
#   - expected_columns (STRING, comma-separated)
#   - actual_columns   (STRING, comma-separated)
# It should log a message showing expected vs actual columns,
# then fail if they don't match (hint: use io.kestra.plugin.core.flow.If
# + io.kestra.plugin.core.execution.Fail).
#
# Your solution here:
challenge_flow = {
    "id": "check-schema",
    "namespace": "data.validation",
    "inputs": [
        # TODO: add your inputs here
    ],
    "tasks": [
        # TODO: add Log and If+Fail tasks here
    ],
}
print(yaml.dump(challenge_flow, default_flow_style=False, sort_keys=False))


---
## Day 7 key concepts recap

| Concept | What to remember |
|---|---|
| Subflow | `io.kestra.plugin.core.flow.Subflow` — call a child flow with typed inputs |
| `wait: true` | Parent blocks until child completes (default is fire-and-forget) |
| `transmitFailed` | Set `true` to propagate child failures up to the parent |
| Accessing child outputs | `{{ outputs.<task_id>.outputs.<output_id> }}` |
| Namespace hierarchy | Dot-separated paths: `data.ingestion`, `data.validation`, etc. |
| Namespace files | Shared scripts mounted into script tasks via `namespaceFiles: enabled: true` |
| `{{ flow.namespace }}` | Pebble variable exposing the current flow's namespace at runtime |
| Pebble filters | `| split('/') | last` — chainable transformations on values |

> **Tip:** Design subflows like functions: a single responsibility, typed inputs, and predictable outputs.

---
## What's next
**Day 8** → Secrets, Variables, and Environment Configuration — store credentials securely, define namespace-level variables, and parameterize flows for dev/staging/prod.

Mark Day 7 complete in your [tracker](../index.html).
